In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer, InputExample, losses, models, util
from sentence_transformers.readers import InputExample
from torch.utils.data import DataLoader
import torch
import torch.nn.functional as F

In [ ]:
df = pd.read_csv("2022_Patient_level_linkage_withIDs.csv", dtype="string")

In [ ]:
df['Match Status'] = df['Match Status'].replace({'Match': '1', 'Non-Match': '0'})
df['Match Status'] = df['Match Status'].astype(float)
df['Overall Similarity'] = df['Overall Similarity'].astype(float)
df.rename(columns={'Match Status': 'labels'}, inplace=True)

In [ ]:
df['record1 Sex'] = df['record1 Sex'].replace({'1': 'Male', '2': 'Female'})
df['record2 Sex'] = df['record2 Sex'].replace({'1': 'Male', '2': 'Female'})
df = df.replace('Unknown', '')

In [ ]:
from sentence_transformers import SentenceTransformer, models


base_model_name = "roberta-base"


word_embedding_model = models.Transformer(
    model_name_or_path=base_model_name,
    model_args={'add_pooling_layer': False}
)


pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False,
    include_prompt=False
)


model = SentenceTransformer(modules=[word_embedding_model, pooling_model])


print(model)


In [ ]:
def serialize_record(record_dict):
    serialized_str = ""
    for col_name, value in record_dict.items():
        serialized_str += f" {value}"
    return serialized_str.strip().lower()

In [ ]:

def create_input_examples(df):
    examples = []
    for _, row in df.iterrows():
        record1_dict = {
            "FirstName": row["record1 First Name"],
            "MiddleName": row["record1 Middle Name"],
            "LastName": row["record1 Last Name"],
            "BirthDate": row["record1 Date of Birth"],
            "Sex": row["record1 Sex"],
        }
        record2_dict = {
            "FirstName": row["record2 First Name"],
            "MiddleName": row["record2 Middle Name"],
            "LastName": row["record2 Last Name"],
            "BirthDate": row["record2 Date of Birth"],
            "Sex": row["record2 Sex"],
        }
        
        serialized_r1 = serialize_record(record1_dict)
        serialized_r2 = serialize_record(record2_dict)
        label = row["Overall Similarity"]
        
        examples.append(InputExample(texts=[serialized_r1, serialized_r2], label=label))
    return examples


input_examples = create_input_examples(df)
print(f"Total examples: {len(input_examples)}")


In [ ]:
import torch
import numpy as np
import random


torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
from sentence_transformers import SentencesDataset


train_dataset = SentencesDataset(input_examples, model=model)


train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64)


In [ ]:
from sentence_transformers import losses
from sentence_transformers.losses import ContrastiveLoss
from sentence_transformers.readers import InputExample


train_loss = losses.CosineSimilarityLoss(model=model)


num_epochs = 5
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)


from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator





model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,


    show_progress_bar=True 
)



In [ ]:

model_save_path = f"finetuned-{base_model_name}-ForBlocking"
model.save(model_save_path)

print(f"Model saved to {model_save_path}")
